In [2]:
"""
Test Script 1 — test_shared.py
Tests the shared.py module: config constants, embed() function, get_index().
Mocks boto3 and Pinecone to avoid needing live cloud credentials.
"""
import sys
import types
import json
import unittest

# ── Mock boto3, pinecone, dotenv BEFORE importing shared ──────────────
mock_boto3 = types.ModuleType("boto3")
mock_response_body = type("Body", (), {
    "read": lambda self: json.dumps({"embedding": [0.1] * 1024}).encode()
})()
mock_bedrock = type("BedrockClient", (), {
    "invoke_model": lambda self, **kw: {"body": mock_response_body}
})()
mock_boto3.client = lambda *a, **kw: mock_bedrock
sys.modules["boto3"] = mock_boto3

mock_dotenv = types.ModuleType("dotenv")
mock_dotenv.load_dotenv = lambda *a, **kw: None
sys.modules["dotenv"] = mock_dotenv

mock_pinecone_mod = types.ModuleType("pinecone")
class MockIndex:
    def query(self, **kw): return {"matches": []}
    def upsert(self, **kw): pass
class MockPinecone:
    def __init__(self, **kw): pass
    def Index(self, name): return MockIndex()
    def list_indexes(self): return []
mock_pinecone_mod.Pinecone = MockPinecone
sys.modules["pinecone"] = mock_pinecone_mod

# Now import shared
sys.path.insert(0, "/home/claude/workday-cr-test-gen")

import importlib
import shared
importlib.reload(shared)


class TestSharedConfig(unittest.TestCase):
    """Test configuration constants are set correctly."""

    def test_bedrock_region_has_value(self):
        self.assertTrue(len(shared.BEDROCK_REGION) > 0)

    def test_index_name_has_value(self):
        self.assertTrue(len(shared.INDEX_NAME) > 0)

    def test_embedding_model_is_titan(self):
        self.assertIn("titan-embed-text", shared.EMBEDDING_MODEL)

    def test_llm_model_is_nova(self):
        self.assertIn("nova", shared.LLM_MODEL)

    def test_embedding_dimension_is_1024(self):
        result = shared.embed("test")
        self.assertEqual(len(result), 1024)


class TestEmbedFunction(unittest.TestCase):
    """Test the embed() helper function."""

    def test_returns_list_of_floats(self):
        result = shared.embed("hello world")
        self.assertIsInstance(result, list)
        self.assertIsInstance(result[0], float)

    def test_returns_1024_dimensions(self):
        result = shared.embed("test input")
        self.assertEqual(len(result), 1024)

    def test_truncates_long_input(self):
        """Input longer than 8000 chars should not raise an error."""
        long_text = "x" * 20000
        result = shared.embed(long_text)
        self.assertEqual(len(result), 1024)

    def test_handles_empty_string(self):
        result = shared.embed("")
        self.assertEqual(len(result), 1024)


class TestGetIndex(unittest.TestCase):
    """Test the get_index() function."""

    def test_returns_index_object(self):
        idx = shared.get_index()
        self.assertIsNotNone(idx)

    def test_index_has_query_method(self):
        idx = shared.get_index()
        self.assertTrue(hasattr(idx, "query"))

    def test_index_has_upsert_method(self):
        idx = shared.get_index()
        self.assertTrue(hasattr(idx, "upsert"))


if __name__ == "__main__":
    unittest.main()

usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [--durations N] [-f]
                             [-c] [-b] [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument 'c:\\Users\\zensh\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3493af77c5b7dcbc1a7c7d1bd5a980b0b843dee45.json'


SystemExit: 2